In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split, Subset
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_names = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

dataset = Subset(dataset, range(300))

train_size = 240
val_size = 60

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

def get_model(name):

    if name == "vgg":
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(4096, 10)

    elif name == "resnet":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, 10)

    elif name == "googlenet":
        model = models.googlenet(
            weights=models.GoogLeNet_Weights.DEFAULT,
            aux_logits=True
        )
        model.fc = nn.Linear(model.fc.in_features, 10)
        model.aux1.fc2 = nn.Linear(model.aux1.fc2.in_features, 10)
        model.aux2.fc2 = nn.Linear(model.aux2.fc2.in_features, 10)

    return model

def train_and_evaluate(model, name, epochs=2):

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    train_accs = []
    val_accs = []

    for epoch in range(epochs):

        model.train()

        correct = 0
        total = 0

        for inputs, labels in train_loader:

            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)

            if isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = 100 * correct / total
        train_accs.append(train_acc)

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for inputs, labels in val_loader:

                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)

                if isinstance(outputs, tuple):
                    outputs = outputs[0]

                _, predicted = torch.max(outputs, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        val_accs.append(val_acc)

        print(
            f"{name} Epoch {epoch + 1}/{epochs} - "
            f"Train Acc: {train_acc:.2f}% - "
            f"Val Acc: {val_acc:.2f}%"
        )

    return model, train_accs, val_accs

results = {}
trained_models = {}

for model_name in ["vgg", "resnet", "googlenet"]:

    print(f"\nTraining {model_name.upper()}...")

    model = get_model(model_name)

    trained_model, train_acc, val_acc = train_and_evaluate(
        model,
        model_name.upper(),
        epochs=2
    )

    results[model_name] = (train_acc, val_acc)
    trained_models[model_name] = trained_model

plt.figure(figsize=(10, 6))

for name, (train_acc, val_acc) in results.items():
    plt.plot(
        range(1, len(val_acc) + 1),
        val_acc,
        marker='o',
        label=f'{name.upper()} Validation Accuracy'
    )

plt.title("VGG16 vs ResNet18 vs GoogLeNet")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.legend()
plt.grid(True)
plt.show()

print("\nFinal Validation Accuracy")

for name, (train_acc, val_acc) in results.items():
    print(f"{name.upper()}: {val_acc[-1]:.2f}%")

sample_image, sample_label = dataset[0]

plt.figure(figsize=(4, 4))

image_display = sample_image.permute(1, 2, 0)
image_display = image_display * 0.5 + 0.5
image_display = torch.clamp(image_display, 0, 1)

plt.imshow(image_display)
plt.title(f"Actual: {class_names[sample_label]}")
plt.axis('off')
plt.show()

sample_input = sample_image.unsqueeze(0).to(device)

print("\nPrediction Results")

for model_name, model in trained_models.items():

    model.eval()

    with torch.no_grad():

        output = model(sample_input)

        if isinstance(output, tuple):
            output = output[0]

        prediction = torch.argmax(output, dim=1).item()

    print(
        f"{model_name.upper():<10} => "
        f"{class_names[prediction]}"
    )

 76%|███████▋  | 130M/170M [29:39<08:22, 80.0kB/s]